# Module 5 — Portfolio Construction

## Overview

M4 identified **18 STRONG factors** across three implementation
tiers. M5 translates these signal quality ratings into actual
portfolio weights using three complementary construction methods:

| Portfolio | Method | Purpose |
|-----------|--------|---------|
| **Equal-weight (EW)** | $w_j = 1/N$ for all STRONG factors | Benchmark - no optimisation |
| **Mean-Variance (MVO)** | Maximise risk-adjusted return with turnover penalty | Primary - optimised |
| **Risk-Parity (RP)** | Weight by inverse factor volatility | Robustness - no return estimates |

The EW benchmark answers: how much does optimisation add over
simply holding all strong factors equally?
The MVO portfolio is our primary result.
The RP portfolio is robust to expected return estimation error.

## From Factor Portfolios to Stock Portfolios

Each of the 18 STRONG factors is already a long-short portfolio
of stocks (constructed in M1). The M5 portfolio is a
**portfolio of factor portfolios** - a linear combination of
the 18 long-short factor returns:

$$R_{p,t} = \sum_{j=1}^{J} w_j F_{j,t}$$

where $w_j$ is the weight on factor $j$ and $F_{j,t}$ is its
month-$t$ long-short return from M1.

## Mean-Variance Optimisation

The MVO portfolio solves:

$$\max_{\mathbf{w}} \; \mathbf{w}^\top \boldsymbol{\mu}
  - \frac{\gamma}{2} \mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w}
  - \lambda \|\mathbf{w} - \mathbf{w}_{t-1}\|_1$$

where:
- $\boldsymbol{\mu}$ = expected factor returns (from M4 signal scores)
- $\boldsymbol{\Sigma}$ = factor return covariance matrix (from M1)
- $\gamma$ = risk aversion parameter (set to 2.0)
- $\lambda$ = turnover penalty (scaled by M4 implementation tier)

**Turnover penalty by tier:**
- Tier 1 (STRONG/HIGH): $\lambda = 0.001$ -- low penalty, slow-moving
- Tier 2 (STRONG/MEDIUM): $\lambda = 0.005$ -- moderate penalty
- Tier 3 (STRONG/LOW): $\lambda = 0.020$ -- heavy penalty, fast-moving

**Constraints:**
- $\sum_j w_j = 1$ (fully invested in factor space)
- $w_j \geq 0$ (long-only factor weights — each factor is already
  long-short at the stock level)
- $w_j \leq 0.30$ (maximum 30% in any single factor)

## Inputs and Outputs

**Inputs:**
- `outputs/M1/factor_returns_41.parquet` — factor return series
- `outputs/M4/signal_dashboard.csv` — signal scores and tiers
- `outputs/M4/ic_results.csv` — ICIR for expected return proxy

**Outputs** (saved to `outputs/M5/`):
- `portfolio_weights.csv` — static weights for all three portfolios
- `portfolio_returns.parquet` — monthly returns for all three
- `portfolio_stats.csv` — Sharpe, drawdown, turnover statistics
- `figures/` — cumulative return plots, weight charts

In [5]:
# Imports and Configuration

import pandas as pd
import numpy as np
from scipy.optimize import minimize
import plotly.graph_objects as go
import plotly.io as pio
import os, warnings
warnings.filterwarnings('ignore')

pio.templates.default = "plotly_white"
pio.renderers.default = "notebook"

# ── Paths ─────────────────────────────────────────────────────
PROJECT_ROOT = (r"C:\Users\amosa\Documents\My Graduate School"
                r"\SPRING 2026\Courses\AMS520_Machine Learning in Finance"
                r"\Project\General ML4Trading Resource"
                r"\Personal_Fundamental Factor Investing"
                r"\fundamental-factor-investing")
M1_OUT  = os.path.join(PROJECT_ROOT, "outputs", "M1")
M4_OUT  = os.path.join(PROJECT_ROOT, "outputs", "M4")
OUT_DIR = os.path.join(PROJECT_ROOT, "outputs", "M5")
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# ── Factor families ────────────────────────────────────────────
CHAR_FAMILIES = {
    "Value":         ["BEME","E2P","CF2P","D2P","S2P","A2ME"],
    "Profitability": ["PROF","ROE","ROA","OP","PM","PCM","RNA"],
    "Investment":    ["Investment","NOA","DPI2A","NI","OA","AC"],
    "Momentum":      ["r12_2","r2_1","r12_7","r36_13",
                      "LT_Rev","SUV","Rel2High"],
    "Risk":          ["Beta","IdioVol","Variance",
                      "Spread","LTurnover","LME"],
    "Other":         ["Q","C","AT","ATO","CTO",
                      "D2A","FC2Y","Lev","SGA2S"],
}
ALL_41 = [c for fam in CHAR_FAMILIES.values() for c in fam]

FAMILY_COLORS = {
    "Value":         "#1f77b4",
    "Profitability": "#2ca02c",
    "Investment":    "#ff7f0e",
    "Momentum":      "#d62728",
    "Risk":          "#9467bd",
    "Other":         "#8c564b",
}

# ── MVO parameters ────────────────────────────────────────────
GAMMA       = 2.0    # risk aversion
MAX_WEIGHT  = 0.30   # maximum weight per factor

# Turnover penalty by implementation tier
LAMBDA_BY_TIER = {
    'Tier 1': 0.001,   # STRONG/HIGH  — slow-moving
    'Tier 2': 0.005,   # STRONG/MEDIUM — moderate
    'Tier 3': 0.020,   # STRONG/LOW   — fast-moving
}

print(f"Output directory: {OUT_DIR}")
print(f"MVO parameters:   γ={GAMMA}, max_weight={MAX_WEIGHT:.0%}")

Output directory: C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS520_Machine Learning in Finance\Project\General ML4Trading Resource\Personal_Fundamental Factor Investing\fundamental-factor-investing\outputs\M5
MVO parameters:   γ=2.0, max_weight=30%


In [6]:
# Load data and define investment universe

print("Loading data...")

# Factor returns from M1
factor_returns = pd.read_parquet(
    os.path.join(M1_OUT, "factor_returns_41.parquet"))
factor_returns.index = (pd.to_datetime(factor_returns.index)
                        + pd.offsets.MonthEnd(0))
factor_returns = factor_returns[ALL_41].copy()

# Signal dashboard from M4
dashboard = pd.read_csv(
    os.path.join(M4_OUT, "signal_dashboard.csv"),
    index_col=0)

# IC results from M4 (ICIR used as expected return proxy)
ic_results = pd.read_csv(
    os.path.join(M4_OUT, "ic_results.csv"),
    index_col=0)

T = len(factor_returns)
print(f"Factor returns:  {factor_returns.shape}")
print(f"Date range:      {factor_returns.index.min().date()} — "
      f"{factor_returns.index.max().date()}")

# ── Define investment universe: STRONG factors only ───────────
strong_factors = dashboard[
    dashboard['Rating'] == 'STRONG'].index.tolist()

# Confirm tier assignments
print(f"\nInvestment universe: {len(strong_factors)} STRONG factors")
print(f"\n{'─'*55}")
print(f"{'Factor':<16} {'Family':<15} {'Tier':<10} {'ICIR':>7}")
print(f"{'─'*55}")

for char in strong_factors:
    tier   = dashboard.loc[char, 'Tier']
    family = next(f for f, cs in CHAR_FAMILIES.items()
                  if char in cs)
    icir   = ic_results.loc[char, 'ICIR (ann)']
    print(f"  {char:<16} {family:<15} {tier:<10} {icir:>7.3f}")

# Factor returns for strong factors only
fr_strong = factor_returns[strong_factors].copy()

# Handle nulls: forward-fill then drop remaining
fr_strong = fr_strong.fillna(method='ffill').dropna()
print(f"\nClean factor returns: {fr_strong.shape}")

Loading data...
Factor returns:  (696, 41)
Date range:      1967-01-31 — 2024-12-31

Investment universe: 18 STRONG factors

───────────────────────────────────────────────────────
Factor           Family          Tier          ICIR
───────────────────────────────────────────────────────
  BEME             Value           Tier 1      -3.677
  E2P              Value           Tier 2      -0.647
  CF2P             Value           Tier 2      -1.342
  S2P              Value           Tier 1      -2.408
  A2ME             Value           Tier 1      -3.347
  PROF             Profitability   Tier 1       0.525
  NI               Investment      Tier 2      -1.290
  AC               Investment      Tier 2      -0.381
  r2_1             Momentum        Tier 2      -1.229
  r36_13           Momentum        Tier 3       0.670
  SUV              Momentum        Tier 3       4.426
  Rel2High         Momentum        Tier 2      10.233
  Variance         Risk            Tier 2      -1.278
  Spread 

---

## Expected Returns and Covariance Estimation

### Expected Returns ($\boldsymbol{\mu}$)

We use the **annualized ICIR** from M4 as a proxy for expected
factor returns. The ICIR measures risk-adjusted signal strength
- it is the Sharpe ratio of the monthly IC time series. A factor
with a high ICIR has consistently predicted returns well and
should receive a higher expected return in the optimisation.

We scale the ICIR to monthly return units using the relationship:

$$\mu_j = \frac{\overline{\text{IC}}_j \times \sigma_{r}}
  {\sqrt{12}} \times \text{sign}(\overline{\text{IC}}_j)$$

where $\sigma_r$ is the cross-sectional standard deviation of
returns (approximately 10% per month for our universe).
This is the **fundamental law of active management** (Grinold
1989): $\text{IR} = \text{IC} \times \sqrt{\text{breadth}}$.

In practice, we use the full-sample mean factor return as the
primary expected return estimate, with the ICIR as a shrinkage
signal - factors with higher ICIR receive less shrinkage toward
the cross-sectional mean.

### Covariance Matrix ($\boldsymbol{\Sigma}$)

We estimate the factor return covariance matrix using the
**Ledoit-Wolf shrinkage estimator**, which produces a
well-conditioned matrix even when the number of factors is
large relative to the number of observations. The shrinkage
pulls sample covariance estimates toward a structured target,
reducing estimation error.

With $T = 696$ months and $J = 18$ factors, the sample
covariance matrix is well-identified ($T \gg J$), so shrinkage
has a modest effect but improves out-of-sample performance.

In [7]:
# Expected returns and Covariance

from sklearn.covariance import LedoitWolf

print("Estimating expected returns and covariance matrix...\n")

# ── Expected returns ──────────────────────────────────────────
# Use full-sample mean factor return as base estimate
# Shrink toward zero proportionally to ICIR
# High ICIR → less shrinkage (more confident in the signal)

mu_raw = fr_strong.mean()   # monthly mean return

# ICIR-based shrinkage: shrinkage factor = 1 - |ICIR| / max(|ICIR|)
# Factors with highest ICIR get least shrinkage
icir_abs = ic_results.loc[strong_factors, 'ICIR (ann)'].abs()
icir_norm = icir_abs / icir_abs.max()          # normalize to [0,1]
shrinkage = 1 - icir_norm                       # high ICIR → low shrinkage

mu = mu_raw * (1 - shrinkage * 0.3)            # max 30% shrinkage

print("Expected monthly returns (annualized %):")
print(f"{'Factor':<16} {'Raw μ%':>9} {'Shrunk μ%':>10} {'ICIR':>8}")
print("─" * 48)
for char in strong_factors:
    fam = next(f for f, cs in CHAR_FAMILIES.items() if char in cs)
    print(f"  {char:<16} "
          f"{mu_raw[char]*12*100:>9.2f} "
          f"{mu[char]*12*100:>10.2f} "
          f"{icir_abs[char]:>8.3f}")

# ── Covariance matrix (Ledoit-Wolf shrinkage) ─────────────────
lw = LedoitWolf()
lw.fit(fr_strong.values)
Sigma    = pd.DataFrame(lw.covariance_,
                        index=strong_factors,
                        columns=strong_factors)
lw_alpha = lw.shrinkage_

print(f"\nLedoit-Wolf shrinkage coefficient: {lw_alpha:.4f}")
print(f"Covariance matrix: {Sigma.shape}")
print(f"Condition number:  {np.linalg.cond(Sigma.values):.2f}")
print(f"(should be << 8279 from M2 — LW shrinkage improves conditioning)")

Estimating expected returns and covariance matrix...

Expected monthly returns (annualized %):
Factor              Raw μ%  Shrunk μ%     ICIR
────────────────────────────────────────────────
  BEME                 63.48      51.28    3.677
  E2P                  12.00       8.63    0.647
  CF2P                 22.54      16.66    1.342
  S2P                  41.16      31.72    2.408
  A2ME                 58.56      46.74    3.347
  PROF                  6.32       4.52    0.525
  NI                    7.65       5.64    1.290
  AC                    8.58       6.10    0.381
  r2_1                 14.63      10.77    1.229
  r36_13               10.43       7.50    0.670
  SUV                  77.30      64.14    4.426
  Rel2High            205.94     205.94   10.233
  Variance            -10.59      -7.81    1.278
  Spread               23.77      19.35    3.881
  LTurnover            48.97      36.85    1.787
  LME                   8.97       6.58    1.144
  Q                    62

---

## Portfolio 1 - Equal Weight

In [8]:
print("=== Portfolio 1: Equal Weight ===\n")

J   = len(strong_factors)
w_ew = pd.Series(1.0 / J, index=strong_factors)

print(f"Number of factors: {J}")
print(f"Weight per factor: {1/J:.4f} ({1/J:.1%})")
print(f"Sum of weights:    {w_ew.sum():.6f}")

# Portfolio returns
ret_ew = (fr_strong * w_ew).sum(axis=1)

# Statistics
ann_ret_ew = ret_ew.mean() * 12
ann_vol_ew = ret_ew.std()  * np.sqrt(12)
sharpe_ew  = ann_ret_ew / ann_vol_ew

# Max drawdown
cum_ew     = (1 + ret_ew).cumprod()
roll_max   = cum_ew.cummax()
mdd_ew     = ((cum_ew - roll_max) / roll_max).min()

print(f"\nEqual-Weight Portfolio Statistics:")
print(f"  Annualized return:    {ann_ret_ew*100:.2f}%")
print(f"  Annualized volatility:{ann_vol_ew*100:.2f}%")
print(f"  Sharpe ratio:         {sharpe_ew:.3f}")
print(f"  Max drawdown:         {mdd_ew*100:.2f}%")

=== Portfolio 1: Equal Weight ===

Number of factors: 18
Weight per factor: 0.0556 (5.6%)
Sum of weights:    1.000000

Equal-Weight Portfolio Statistics:
  Annualized return:    38.24%
  Annualized volatility:6.89%
  Sharpe ratio:         5.546
  Max drawdown:         -4.06%


---

## Portfolio 2 - Mean Variance Optimization

In [9]:
print("=== Portfolio 2: Mean-Variance Optimisation ===\n")
print(f"γ = {GAMMA}  (risk aversion)")
print(f"Turnover penalties by tier: {LAMBDA_BY_TIER}\n")

# ── Turnover penalty vector ───────────────────────────────────
lambda_vec = np.array([
    LAMBDA_BY_TIER.get(dashboard.loc[c, 'Tier'], 0.005)
    for c in strong_factors
])

print("Turnover penalties per factor:")
for char, lam in zip(strong_factors, lambda_vec):
    tier = dashboard.loc[char, 'Tier']
    print(f"  {char:<16} {tier:<10}  λ = {lam:.3f}")

# ── MVO objective function ────────────────────────────────────
mu_arr    = mu[strong_factors].values
Sigma_arr = Sigma.values

def mvo_objective(w, w_prev, mu, Sigma, gamma, lambda_vec):
    """
    Negative of: w'μ - (γ/2) w'Σw - λ||w - w_prev||_1
    (negative because scipy minimizes)
    """
    ret      = w @ mu
    risk     = 0.5 * gamma * (w @ Sigma @ w)
    turnover = np.sum(lambda_vec * np.abs(w - w_prev))
    return -(ret - risk - turnover)

def mvo_gradient(w, w_prev, mu, Sigma, gamma, lambda_vec):
    grad_ret  = mu
    grad_risk = gamma * Sigma @ w
    grad_tc   = lambda_vec * np.sign(w - w_prev)
    return -(grad_ret - grad_risk - grad_tc)

# ── Static MVO (full-sample, single optimisation) ────────────
# w_prev = equal weight (no prior position)
w_prev = np.ones(J) / J

constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1}]
bounds      = [(0.0, MAX_WEIGHT)] * J

result = minimize(
    fun      = mvo_objective,
    x0       = w_prev,
    args     = (w_prev, mu_arr, Sigma_arr,
                GAMMA, lambda_vec),
    jac      = mvo_gradient,
    method   = 'SLSQP',
    bounds   = bounds,
    constraints = constraints,
    options  = {'ftol': 1e-12, 'maxiter': 1000}
)

if not result.success:
    print(f"WARNING: Optimisation did not converge: {result.message}")
else:
    print(f"\nOptimisation converged in {result.nit} iterations")

w_mvo = pd.Series(result.x, index=strong_factors)

# Round tiny weights to zero
w_mvo[w_mvo < 1e-4] = 0.0
w_mvo = w_mvo / w_mvo.sum()  # renormalize

print(f"\nMVO Portfolio Weights:")
print(f"{'Factor':<16} {'Weight':>8} {'Tier':<10} {'Family'}")
print("─" * 50)
for char in strong_factors:
    if w_mvo[char] > 0.001:
        tier   = dashboard.loc[char, 'Tier']
        family = next(f for f, cs in CHAR_FAMILIES.items()
                      if char in cs)
        print(f"  {char:<16} {w_mvo[char]:>8.4f} "
              f"{tier:<10} {family}")

n_active = (w_mvo > 0.001).sum()
print(f"\nActive factors: {n_active}/{J}")
print(f"Sum of weights: {w_mvo.sum():.6f}")
print(f"Max weight:     {w_mvo.max():.4f} ({w_mvo.idxmax()})")
print(f"Min weight:     {w_mvo[w_mvo>0.001].min():.4f} "
      f"({w_mvo[w_mvo>0.001].idxmin()})")

# Portfolio returns
ret_mvo    = (fr_strong * w_mvo).sum(axis=1)
ann_ret_mvo = ret_mvo.mean() * 12
ann_vol_mvo = ret_mvo.std()  * np.sqrt(12)
sharpe_mvo  = ann_ret_mvo / ann_vol_mvo
cum_mvo     = (1 + ret_mvo).cumprod()
mdd_mvo     = ((cum_mvo - cum_mvo.cummax()) / cum_mvo.cummax()).min()

print(f"\nMVO Portfolio Statistics:")
print(f"  Annualized return:    {ann_ret_mvo*100:.2f}%")
print(f"  Annualized volatility:{ann_vol_mvo*100:.2f}%")
print(f"  Sharpe ratio:         {sharpe_mvo:.3f}")
print(f"  Max drawdown:         {mdd_mvo*100:.2f}%")

=== Portfolio 2: Mean-Variance Optimisation ===

γ = 2.0  (risk aversion)
Turnover penalties by tier: {'Tier 1': 0.001, 'Tier 2': 0.005, 'Tier 3': 0.02}

Turnover penalties per factor:
  BEME             Tier 1      λ = 0.001
  E2P              Tier 2      λ = 0.005
  CF2P             Tier 2      λ = 0.005
  S2P              Tier 1      λ = 0.001
  A2ME             Tier 1      λ = 0.001
  PROF             Tier 1      λ = 0.001
  NI               Tier 2      λ = 0.005
  AC               Tier 2      λ = 0.005
  r2_1             Tier 2      λ = 0.005
  r36_13           Tier 3      λ = 0.020
  SUV              Tier 3      λ = 0.020
  Rel2High         Tier 2      λ = 0.005
  Variance         Tier 2      λ = 0.005
  Spread           Tier 3      λ = 0.020
  LTurnover        Tier 2      λ = 0.005
  LME              Tier 2      λ = 0.005
  Q                Tier 1      λ = 0.001
  Lev              Tier 1      λ = 0.001

Optimisation converged in 27 iterations

MVO Portfolio Weights:
Factor      

---

## Portfolio 3 - Risk Parity

In [10]:
print("=== Portfolio 3: Risk Parity ===\n")
print("Weights proportional to inverse factor volatility.")
print("No expected return estimates required.\n")

# Factor volatilities (annualized)
factor_vols = fr_strong.std() * np.sqrt(12)

# Inverse volatility weights
inv_vol  = 1.0 / factor_vols
w_rp     = inv_vol / inv_vol.sum()

print(f"{'Factor':<16} {'Vol (ann%)':>11} {'Weight':>8}")
print("─" * 40)
for char in strong_factors:
    print(f"  {char:<16} "
          f"{factor_vols[char]*100:>11.2f} "
          f"{w_rp[char]:>8.4f}")

print(f"\nSum of weights: {w_rp.sum():.6f}")

# Portfolio returns
ret_rp     = (fr_strong * w_rp).sum(axis=1)
ann_ret_rp = ret_rp.mean() * 12
ann_vol_rp = ret_rp.std()  * np.sqrt(12)
sharpe_rp  = ann_ret_rp / ann_vol_rp
cum_rp     = (1 + ret_rp).cumprod()
mdd_rp     = ((cum_rp - cum_rp.cummax()) / cum_rp.cummax()).min()

print(f"\nRisk Parity Portfolio Statistics:")
print(f"  Annualized return:    {ann_ret_rp*100:.2f}%")
print(f"  Annualized volatility:{ann_vol_rp*100:.2f}%")
print(f"  Sharpe ratio:         {sharpe_rp:.3f}")
print(f"  Max drawdown:         {mdd_rp*100:.2f}%")

=== Portfolio 3: Risk Parity ===

Weights proportional to inverse factor volatility.
No expected return estimates required.

Factor            Vol (ann%)   Weight
────────────────────────────────────────
  BEME                   17.75   0.0496
  E2P                    16.66   0.0529
  CF2P                   18.60   0.0474
  S2P                    18.03   0.0489
  A2ME                   19.16   0.0460
  PROF                    9.43   0.0934
  NI                     14.32   0.0615
  AC                      7.65   0.1151
  r2_1                   17.60   0.0501
  r36_13                 16.59   0.0531
  SUV                    18.94   0.0465
  Rel2High               25.48   0.0346
  Variance               30.52   0.0289
  Spread                 10.53   0.0837
  LTurnover              23.66   0.0372
  LME                    20.43   0.0431
  Q                      18.18   0.0485
  Lev                    14.81   0.0595

Sum of weights: 1.000000

Risk Parity Portfolio Statistics:
  Annualized re

---

## Portfolio Comparison

We compare the three portfolios against each other across
five dimensions:

- **Sharpe ratio** -- risk-adjusted return (primary metric)
- **Annualized return** -- raw performance
- **Annualized volatility** -- total risk
- **Maximum drawdown** -- worst peak-to-trough experience
- **Concentration** -- effective number of factors
  ($= 1 / \sum_j w_j^2$, the inverse Herfindahl index)

The **effective N** measures diversification: equal-weight
gives effective N = J (maximum diversification), while a
portfolio with all weight in one factor gives effective N = 1.

If MVO does not outperform EW on Sharpe ratio, it means
the expected return estimates ($\boldsymbol{\mu}$) are not
adding value - a common finding that motivates using RP
as the primary method in practice.

In [12]:
# Portfolio comparison

print("=== Portfolio Comparison: In-Sample Portfolio Performance ===\n")

portfolios = {
    'Equal Weight': {
        'returns': ret_ew,
        'weights': w_ew,
        'ann_ret': ann_ret_ew,
        'ann_vol': ann_vol_ew,
        'sharpe':  sharpe_ew,
        'mdd':     mdd_ew,
    },
    'MVO': {
        'returns': ret_mvo,
        'weights': w_mvo,
        'ann_ret': ann_ret_mvo,
        'ann_vol': ann_vol_mvo,
        'sharpe':  sharpe_mvo,
        'mdd':     mdd_mvo,
    },
    'Risk Parity': {
        'returns': ret_rp,
        'weights': w_rp,
        'ann_ret': ann_ret_rp,
        'ann_vol': ann_vol_rp,
        'sharpe':  sharpe_rp,
        'mdd':     mdd_rp,
    },
}

# Effective N (inverse Herfindahl)
for name, p in portfolios.items():
    w = p['weights']
    p['eff_n'] = 1.0 / (w ** 2).sum()

print(f"{'─'*70}")
print(f"{'Metric':<25} {'Equal Weight':>14} {'MVO':>14} "
      f"{'Risk Parity':>14}")
print(f"{'─'*70}")

metrics = [
    ('Ann. Return (%)',    'ann_ret', 100,  '.2f'),
    ('Ann. Volatility (%)', 'ann_vol', 100, '.2f'),
    ('Sharpe Ratio',       'sharpe',   1,  '.3f'),
    ('Max Drawdown (%)',   'mdd',     100,  '.2f'),
    ('Effective N',        'eff_n',     1,  '.1f'),
]

for label, key, scale, fmt in metrics:
    vals = {n: p[key] * scale for n, p in portfolios.items()}
    row  = f"  {label:<23}"
    for name in ['Equal Weight','MVO','Risk Parity']:
        v = vals[name]
        row += f" {v:>14{fmt}}"
    print(row)

print(f"{'─'*70}")

# Best portfolio on each metric
print(f"\nBest on each metric:")
for label, key, scale, fmt in metrics:
    if key == 'mdd':   # lower is better
        best = min(portfolios, key=lambda n: portfolios[n][key])
    elif key in ['ann_vol']:  # lower is better
        best = min(portfolios, key=lambda n: portfolios[n][key])
    else:              # higher is better
        best = max(portfolios, key=lambda n: portfolios[n][key])
    print(f"  {label:<25}: {best}")

=== Portfolio Comparison: In-Sample Portfolio Performance ===

──────────────────────────────────────────────────────────────────────
Metric                      Equal Weight            MVO    Risk Parity
──────────────────────────────────────────────────────────────────────
  Ann. Return (%)                  38.24         103.96          32.08
  Ann. Volatility (%)               6.89          16.15           6.06
  Sharpe Ratio                     5.546          6.437          5.296
  Max Drawdown (%)                 -4.06         -16.61          -2.80
  Effective N                       18.0            4.7           15.8
──────────────────────────────────────────────────────────────────────

Best on each metric:
  Ann. Return (%)          : MVO
  Ann. Volatility (%)      : Risk Parity
  Sharpe Ratio             : MVO
  Max Drawdown (%)         : MVO
  Effective N              : Equal Weight


In [14]:
# Cumulative Return Plots: in sample 

print("Generating cumulative return plot...")

# Log cumulative returns
log_cum_ew  = np.log1p(ret_ew).cumsum()  * 100
log_cum_mvo = np.log1p(ret_mvo).cumsum() * 100
log_cum_rp  = np.log1p(ret_rp).cumsum()  * 100

fig = go.Figure()

for name, log_cum, color in [
    ('Equal Weight', log_cum_ew,  '#888888'),
    ('MVO',          log_cum_mvo, '#d62728'),
    ('Risk Parity',  log_cum_rp,  '#1f77b4'),
]:
    fig.add_trace(go.Scatter(
        x    = log_cum.index,
        y    = log_cum.values,
        name = name,
        mode = 'lines',
        line = dict(width=2.0, color=color),
        hovertemplate=(
            f"<b>{name}</b><br>"
            "Date: %{x|%b %Y}<br>"
            "Log Cum Return: %{y:.1f}%<extra></extra>"
        )
    ))

fig.add_hline(y=0,
              line=dict(color='black', width=1, dash='dash'),
              opacity=0.4)

# NBER recession shading
RECESSIONS = [
    ('1969-12-01','1970-11-30'), ('1973-11-01','1975-03-31'),
    ('1980-01-01','1980-07-31'), ('1981-07-01','1982-11-30'),
    ('1990-07-01','1991-03-31'), ('2001-03-01','2001-11-30'),
    ('2007-12-01','2009-06-30'), ('2020-02-01','2020-04-30'),
]
for rs, re in RECESSIONS:
    rs, re = pd.Timestamp(rs), pd.Timestamp(re)
    if rs >= log_cum_ew.index.min():
        fig.add_vrect(x0=rs, x1=re, fillcolor='gray',
                      opacity=0.08, layer='below', line_width=0)

fig.update_layout(
    title=dict(
        text=("<b>Portfolio Cumulative Returns — M5</b><br>"
              "<sup>Log cumulative returns | 18 STRONG factors | "
              "Gray = NBER recessions</sup>"),
        font=dict(size=14), x=0.0),
    xaxis=dict(title='Date', showgrid=True,
               gridcolor='rgba(200,200,200,0.3)'),
    yaxis=dict(title='Log Cumulative Return (%)',
               ticksuffix='%', showgrid=True,
               gridcolor='rgba(200,200,200,0.3)'),
    legend=dict(orientation='h', yanchor='bottom',
                y=1.02, xanchor='right', x=1),
    width=1000, height=480,
    margin=dict(l=60, r=60, t=90, b=60),
    hovermode='x unified'
)
fig.show()
fig.write_image(os.path.join(FIG_DIR, "cumulative_returns.png"), scale=2)
print("Saved: cumulative_returns.png")

Generating cumulative return plot...


Saved: cumulative_returns.png


In [16]:
print("Generating weight allocation charts...")

PORTFOLIO_COLORS = {
    'Equal Weight': '#888888',
    'MVO':          '#d62728',
    'Risk Parity':  '#1f77b4',
}

fig = go.Figure()

for name, p in portfolios.items():
    fig.add_trace(go.Bar(
        name         = name,
        x            = strong_factors,
        y            = p['weights'].values * 100,
        marker_color = PORTFOLIO_COLORS[name],
        opacity      = 0.85,
        hovertemplate=(
            f"<b>%{{x}}</b><br>{name}<br>"
            "Weight: %{y:.2f}%<extra></extra>"
        )
    ))

# Add family labels as x-axis annotations using shapes
# Mark family boundaries with vertical lines
family_boundaries = []
idx = 0
for fam, chars in CHAR_FAMILIES.items():
    fam_chars = [c for c in chars if c in strong_factors]
    if len(fam_chars) == 0:
        continue
    # Add annotation at the center of each family group
    center = idx + len(fam_chars) / 2 - 0.5
    family_boundaries.append((center, fam, idx, idx + len(fam_chars)))
    idx += len(fam_chars)

fig.update_layout(
    title=dict(
        text=("<b>Factor Weights Across Three Portfolios</b><br>"
              "<sup>Grey = Equal Weight | Red = MVO | Blue = Risk Parity</sup>"),
        font=dict(size=14), x=0.0),
    xaxis=dict(
        title='Factor',
        tickangle=-45,
        tickfont=dict(size=9)
    ),
    yaxis=dict(
        title='Weight (%)',
        ticksuffix='%',
        showgrid=True,
        gridcolor='rgba(200,200,200,0.3)'
    ),
    barmode='group',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    ),
    width=1100,
    height=500,
    margin=dict(l=60, r=60, t=100, b=130),
)

# Add vertical separator lines between factor families
fam_idx = 0
for fam, chars in CHAR_FAMILIES.items():
    fam_chars = [c for c in chars if c in strong_factors]
    if len(fam_chars) == 0:
        continue
    if fam_idx > 0:
        fig.add_vline(
            x=fam_idx - 0.5,
            line=dict(color='black', width=1, dash='dot'),
            opacity=0.3
        )
    fam_idx += len(fam_chars)

fig.show()
fig.write_image(os.path.join(FIG_DIR, "factor_weights.png"), scale=2)
print("Saved: factor_weights.png")

Generating weight allocation charts...


Saved: factor_weights.png


In [17]:
print("=== Saving M5 outputs ===\n")

# Portfolio weights
weights_df = pd.DataFrame({
    'Equal Weight': portfolios['Equal Weight']['weights'],
    'MVO':          portfolios['MVO']['weights'],
    'Risk Parity':  portfolios['Risk Parity']['weights'],
})
weights_df.to_csv(os.path.join(OUT_DIR, "portfolio_weights.csv"))
print("Saved: portfolio_weights.csv")

# Portfolio returns
returns_df = pd.DataFrame({
    'Equal Weight': portfolios['Equal Weight']['returns'],
    'MVO':          portfolios['MVO']['returns'],
    'Risk Parity':  portfolios['Risk Parity']['returns'],
})
returns_df.to_parquet(os.path.join(OUT_DIR, "portfolio_returns.parquet"))
print("Saved: portfolio_returns.parquet")

# Portfolio statistics
stats_df = pd.DataFrame({
    name: {
        'Ann Return (%)':  p['ann_ret'] * 100,
        'Ann Vol (%)':     p['ann_vol'] * 100,
        'Sharpe':          p['sharpe'],
        'Max DD (%)':      p['mdd'] * 100,
        'Effective N':     p['eff_n'],
    }
    for name, p in portfolios.items()
}).T
stats_df.to_csv(os.path.join(OUT_DIR, "portfolio_stats.csv"))
print("Saved: portfolio_stats.csv")

# File inventory
print(f"\nAll M5 output files:")
for f in sorted(os.listdir(OUT_DIR)):
    full = os.path.join(OUT_DIR, f)
    if os.path.isfile(full):
        print(f"  {f:<45} {os.path.getsize(full)/1024:>7.1f} KB")

# Final summary
print(f"""
{'='*60}
M5 COMPLETE
{'='*60}
Investment universe:  {len(strong_factors)} STRONG factors
  Tier 1 (HIGH):   {(dashboard.loc[strong_factors,'Tier']=='Tier 1').sum()} factors — {', '.join(dashboard.loc[strong_factors][dashboard.loc[strong_factors,'Tier']=='Tier 1'].index.tolist())}
  Tier 2 (MEDIUM): {(dashboard.loc[strong_factors,'Tier']=='Tier 2').sum()} factors
  Tier 3 (LOW):    {(dashboard.loc[strong_factors,'Tier']=='Tier 3').sum()} factors

NOTE: All performance statistics below are IN-SAMPLE
(full period 1967-2024). They reflect the quality of
factor selection and portfolio construction methodology
but are subject to look-ahead bias. Out-of-sample
walk-forward performance is evaluated in M6.

In-sample portfolio statistics:
  {'Portfolio':<20} {'Return%':>9} {'Vol%':>7} {'Sharpe':>8} {'MDD%':>8} {'EffN':>6}
  {'─'*60}""")

for name, p in portfolios.items():
    print(f"  {name:<20} "
          f"{p['ann_ret']*100:>9.2f} "
          f"{p['ann_vol']*100:>7.2f} "
          f"{p['sharpe']:>8.3f} "
          f"{p['mdd']*100:>8.2f} "
          f"{p['eff_n']:>6.1f}")

print(f"""
Key observations:
  - MVO concentrates in {(portfolios['MVO']['weights']>0.001).sum()} factors (eff N={portfolios['MVO']['weights'].pipe(lambda w: 1/(w**2).sum()):.1f})
    reflecting in-sample optimisation toward highest-return factors
  - EW and RP are more diversified and will be more robust
    out-of-sample (evaluated in M6)
  - High in-sample Sharpe ratios across all three methods
    confirm the factor selection pipeline is working correctly

Primary portfolio:  MVO
Feeds into:         M6 -- Walk-Forward Backtesting
{'='*60}
""")

=== Saving M5 outputs ===

Saved: portfolio_weights.csv
Saved: portfolio_returns.parquet
Saved: portfolio_stats.csv

All M5 output files:
  portfolio_returns.parquet                        19.7 KB
  portfolio_stats.csv                               0.3 KB
  portfolio_weights.csv                             1.0 KB

M5 COMPLETE
Investment universe:  18 STRONG factors
  Tier 1 (HIGH):   6 factors — BEME, S2P, A2ME, PROF, Q, Lev
  Tier 2 (MEDIUM): 9 factors
  Tier 3 (LOW):    3 factors

NOTE: All performance statistics below are IN-SAMPLE
(full period 1967-2024). They reflect the quality of
factor selection and portfolio construction methodology
but are subject to look-ahead bias. Out-of-sample
walk-forward performance is evaluated in M6.

In-sample portfolio statistics:
  Portfolio              Return%    Vol%   Sharpe     MDD%   EffN
  ────────────────────────────────────────────────────────────
  Equal Weight             38.24    6.89    5.546    -4.06   18.0
  MVO                     1